# Day 075 — Exercise 5: TalkingHeadPipeline

**What you'll build:** `TalkingHeadPipeline` — full pipeline class binding all four injection functions at construction.

**Why it matters:** The class is the ergonomic interface for the full pipeline: `pipe.speech(text)`, `pipe.frames(img, n)`, `pipe.mux(...)`, `pipe.caption(...)` reads more clearly than passing mock functions on every call.

In [ ]:
from pathlib import Path
import tempfile

def _make_mock_frames(n=5, height=32, width=32):
    import numpy as np
    return [np.zeros((height, width, 3), dtype=np.uint8) for _ in range(n)]
_mock_tts_fn     = lambda text, voice, rate, pitch: b'MP3:' + text[:8].encode()
_mock_capture_fn = lambda img, n: _make_mock_frames(n)
_mock_mux_fn     = lambda video_path, audio_bytes, output_path: (Path(output_path).write_bytes(b'MUX' + bytes(len(audio_bytes))), Path(output_path))[1]
_mock_caption_fn = lambda video_path, text, output_path: (Path(output_path).write_bytes(b'CAP' + bytes(len(text))), Path(output_path))[1]

def generate_speech(text, voice='en-US-AriaNeural', rate='+0%', pitch='+0Hz', tts_fn=None):
    if tts_fn is not None:
        return tts_fn(text, voice, rate, pitch)
    import asyncio, edge_tts
    async def _run():
        comm = edge_tts.Communicate(text, voice, rate=rate, pitch=pitch)
        chunks = []
        async for chunk in comm.stream():
            if chunk['type'] == 'audio': chunks.append(chunk['data'])
        return b''.join(chunks)
    return asyncio.run(_run())

def image_to_frames(image, n_frames, capture_fn=None):
    if capture_fn is not None:
        return capture_fn(image, n_frames)
    import numpy as np
    from PIL import Image as PILImage
    if not isinstance(image, PILImage.Image):
        image = PILImage.open(str(image)).convert('RGB')
    arr = np.array(image)[:, :, ::-1].astype(np.uint8)
    return [arr.copy() for _ in range(n_frames)]

def mux_audio_video(video_path, audio_bytes, output_path, ffmpeg_fn=None):
    if ffmpeg_fn is not None:
        return ffmpeg_fn(video_path, audio_bytes, output_path)
    import subprocess, tempfile
    with tempfile.NamedTemporaryFile(suffix='.mp3', delete=False) as f:
        f.write(audio_bytes); audio_path = f.name
    try:
        out = Path(output_path)
        r = subprocess.run(['ffmpeg','-y','-i',str(video_path),'-i',audio_path,
                            '-c:v','copy','-c:a','aac','-shortest',str(out)],
                           capture_output=True, text=True)
        if r.returncode != 0: raise RuntimeError(r.stderr[-500:])
        return out
    finally:
        Path(audio_path).unlink(missing_ok=True)

def add_captions(video_path, text, output_path,
                 fontsize=24, color='white', ffmpeg_fn=None):
    if ffmpeg_fn is not None:
        return ffmpeg_fn(video_path, text, output_path)
    import subprocess
    safe = text.replace("'", r"\'").replace(':', r'\:')
    out = Path(output_path)
    r = subprocess.run(
        ['ffmpeg','-y','-i',str(video_path),'-vf',
         f"drawtext=text='{safe}':fontsize={fontsize}:"
         f"fontcolor={color}:x=(w-text_w)/2:y=h-text_h-20", str(out)],
        capture_output=True, text=True)
    if r.returncode != 0: raise RuntimeError(r.stderr[-500:])
    return out


## Task

Implement `TalkingHeadPipeline`:

- `__init__`: store `tts_fn`, `capture_fn`, `mux_fn`, `caption_fn` as `self._xxx_fn`
- `speech(text, voice, rate, pitch)`: `return generate_speech(..., tts_fn=self._tts_fn)`
- `frames(image, n_frames)`: `return image_to_frames(..., capture_fn=self._capture_fn)`
- `mux(video_path, audio_bytes, output_path)`: `return mux_audio_video(..., ffmpeg_fn=self._mux_fn)`
- `caption(video_path, text, output_path, fontsize, color)`: `return add_captions(..., ffmpeg_fn=self._caption_fn)`

## Your Implementation

In [ ]:
class TalkingHeadPipeline:
    """Create talking-head videos from text and a face image."""

    def __init__(self, tts_fn=None, capture_fn=None,
                 mux_fn=None, caption_fn=None) -> None:
        raise NotImplementedError

    def speech(self, text: str, voice: str = 'en-US-AriaNeural',
               rate: str = '+0%', pitch: str = '+0Hz') -> bytes:
        raise NotImplementedError

    def frames(self, image, n_frames: int) -> list:
        raise NotImplementedError

    def mux(self, video_path, audio_bytes: bytes, output_path):
        raise NotImplementedError

    def caption(self, video_path, text: str, output_path,
                fontsize: int = 24, color: str = 'white'):
        raise NotImplementedError


In [ ]:
class TalkingHeadPipeline:
    def __init__(self, tts_fn=None, capture_fn=None,
                 mux_fn=None, caption_fn=None):
        self._tts_fn     = tts_fn
        self._capture_fn = capture_fn
        self._mux_fn     = mux_fn
        self._caption_fn = caption_fn

    def speech(self, text, voice='en-US-AriaNeural',
               rate='+0%', pitch='+0Hz'):
        return generate_speech(text, voice=voice, rate=rate, pitch=pitch,
                               tts_fn=self._tts_fn)

    def frames(self, image, n_frames):
        return image_to_frames(image, n_frames, capture_fn=self._capture_fn)

    def mux(self, video_path, audio_bytes, output_path):
        return mux_audio_video(video_path, audio_bytes, output_path,
                               ffmpeg_fn=self._mux_fn)

    def caption(self, video_path, text, output_path,
                fontsize=24, color='white'):
        return add_captions(video_path, text, output_path,
                            fontsize=fontsize, color=color,
                            ffmpeg_fn=self._caption_fn)


## Automated checks

In [ ]:

import tempfile
score, total = 0, 5
try:
    pipe = TalkingHeadPipeline(
        tts_fn=_mock_tts_fn,
        capture_fn=_mock_capture_fn,
        mux_fn=_mock_mux_fn,
        caption_fn=_mock_caption_fn,
    )

    # speech() delegates to generate_speech
    audio = pipe.speech('Day 75 is here!')
    assert isinstance(audio, bytes) and len(audio) > 0
    score += 1; print("✅ speech() returns non-empty bytes")

    # frames() delegates to image_to_frames
    frames = pipe.frames('face.png', 8)
    assert isinstance(frames, list) and len(frames) == 8
    assert frames[0].shape == (32, 32, 3)
    score += 1; print("✅ frames() returns correct list of numpy arrays")

    # mux() delegates to mux_audio_video
    with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as f:
        silent = f.name
    Path(silent).write_bytes(b'SILENT')
    with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as f:
        muxed = f.name
    m = pipe.mux(silent, audio, muxed)
    assert isinstance(m, Path) and m.exists()
    score += 1; print("✅ mux() returns existing Path")

    # caption() delegates to add_captions
    with tempfile.NamedTemporaryFile(suffix='.mp4', delete=False) as f:
        captioned = f.name
    c = pipe.caption(m, 'Day 75 complete!', captioned)
    assert isinstance(c, Path) and c.exists()
    score += 1; print("✅ caption() returns existing Path")

    # injection fns are bound at construction
    called = {}
    def _spy_tts(text, voice, rate, pitch):
        called['tts'] = True
        return b'MP3:spy'
    pipe2 = TalkingHeadPipeline(tts_fn=_spy_tts)
    pipe2.speech('test')
    assert called.get('tts') is True
    score += 1; print("✅ injection fn bound at construction, invoked on call")

except Exception as e:
    print(f"❌ {e}")

print(f"\n{score}/{total} checks passed")
if score == total:
    print("\U0001f389 Exercise complete!")


## Solution

<details><summary>Reveal</summary>

```python
class TalkingHeadPipeline:
    def __init__(self, tts_fn=None, capture_fn=None,
                 mux_fn=None, caption_fn=None):
        self._tts_fn     = tts_fn
        self._capture_fn = capture_fn
        self._mux_fn     = mux_fn
        self._caption_fn = caption_fn

    def speech(self, text, voice='en-US-AriaNeural',
               rate='+0%', pitch='+0Hz'):
        return generate_speech(text, voice=voice, rate=rate, pitch=pitch,
                               tts_fn=self._tts_fn)

    def frames(self, image, n_frames):
        return image_to_frames(image, n_frames, capture_fn=self._capture_fn)

    def mux(self, video_path, audio_bytes, output_path):
        return mux_audio_video(video_path, audio_bytes, output_path,
                               ffmpeg_fn=self._mux_fn)

    def caption(self, video_path, text, output_path,
                fontsize=24, color='white'):
        return add_captions(video_path, text, output_path,
                            fontsize=fontsize, color=color,
                            ffmpeg_fn=self._caption_fn)
```

**Why four methods, each one line?** All logic is in the module-level functions. The class is a binding layer, not a logic layer. A one-line method that calls a module function with a bound injection is the correct level of delegation — not a pass-through, not a reimplementation.

</details>